# Übungen – Block 8: Inventories, Variablen und Facts

Die Übungen bauen auf den Playbooks aus Block 7 auf.

## Musterlösung

## Übung 1: Inventory in YAML

Überführen Sie ein kleines INI-Inventory mit `web` und `db` in YAML.

```yaml
all:
  children:
    web:
      hosts:
        web01:
        web02:
    db:
      hosts:
        db01:
```

## Übung 2: Playbook-Variable

Ersetzen Sie einen festen Paketnamen durch eine Variable.

```yaml
vars:
  package_name: tree

tasks:
  - name: Paket vorhanden
    ansible.builtin.package:
      name: "{{ package_name }}"
      state: present
```

## Übung 3: group_vars

Legen Sie gruppenspezifische Ports oder Paketnamen in `group_vars` ab.

`group_vars/web.yml`:

```yaml
service_port: 8080
```

`group_vars/db.yml`:

```yaml
service_port: 5432
```

## Übung 4: host_vars

Überschreiben Sie für genau einen Host einen gruppenweiten Wert.

`host_vars/web02.yml`:

```yaml
service_port: 8081
```

## Übung 5: Facts anzeigen

Geben Sie Distribution, Hostname und eine geeignete IP-Information per `debug` aus.

```yaml
- name: Facts anzeigen
  ansible.builtin.debug:
    msg:
      - "{{ ansible_distribution }}"
      - "{{ ansible_hostname }}"
      - "{{ ansible_default_ipv4.address | default('n/a') }}"
```

## Übung 6: Facts verwenden

Erzeugen Sie eine Datei, deren Inhalt Hostname und Distribution des Managed Nodes enthält.

```yaml
- name: Systeminfo schreiben
  ansible.builtin.copy:
    dest: /tmp/system-info.txt
    content: |
      Host={{ ansible_hostname }}
      Distribution={{ ansible_distribution }}
```

## Abschlussübung

Parametrisieren Sie ein bestehendes Playbook so, dass Web- und DB-Gruppe unterschiedliche Werte verwenden und hostspezifische Überschreibungen möglich sind.

Beispielstruktur:

```text
inventory.yml
group_vars/
  web.yml
  db.yml
host_vars/
  web02.yml
site.yml
```

`site.yml` verwendet nur Variablennamen; gruppen- und hostspezifische Werte liegen außerhalb des Playbooks.

## Erläuterungen und vollständige Prüfkriterien

### Übung 1 – Inventory
`all.children.web.hosts` und `all.children.db.hosts` bilden die Gruppen in YAML ab. `ansible-inventory -i inventory.yml --graph` zeigt die Gruppenzuordnung, `--list` die aufgelösten Daten. Ein Host kann mehreren Gruppen angehören.

### Übung 2 – Playbook-Variable
Eine Variable wie `package_name` wird über `{{ package_name }}` referenziert. Sie vermeidet doppelt gepflegte Konstanten, aber der Wert muss zum Paketmanager des Zielsystems passen.

### Übung 3 und 4 – Variablenpriorität
`group_vars/web.yml` und `group_vars/db.yml` liefern gruppenspezifische Werte. `host_vars/web02.yml` kann einen Gruppenwert für genau diesen Host überschreiben. Bei Mehrfachgruppenzugehörigkeit muss die Priorität explizit bedacht werden; zusätzliche Play- oder Extra-Variablen können Hostvariablen wiederum übersteuern.

### Übung 5 – Facts
`ansible_facts['distribution']` und `ansible_facts['hostname']` sind nur verfügbar, wenn Facts ermittelt wurden. Für IP-Adressen ist `ansible_facts['default_ipv4']['address']` nur geeignet, wenn ein Default-IPv4-Interface existiert; andernfalls einen passenden Fact auswählen.

### Übung 6 – Facts in Dateien
`copy: content` kann ermittelte Werte in eine Datei schreiben. Bei identischem Inhalt meldet die Wiederholung normalerweise keine Änderung. Der Dateipfad muss beschreibbar sein.

### Abschlussübung – Vollständigkeit
Ein Verzeichnisbaum allein genügt nicht: Das Playbook muss die Werte tatsächlich referenzieren. Prüfen Sie mindestens einen Webhost, einen DB-Host und den Host mit Überschreibung, etwa durch `debug` oder den erzeugten Konfigurationsinhalt.